# R9 · Warm-up: the first XGBoost and early stopping

**Type**: Implement with scikit-learn · **Level**: ●○○ guided · **Time**: about 25 minutes · **core path ★**

## Objective

Train an `XGBClassifier` with **early stopping** on a validation set, see how many rounds it really uses, compare it with the same model at a fixed 1000 rounds and with a forest.

## Dataset

`make_classification(3000, 12 features, 5 informative, flip_y=0.05)`; 60/20/20 split (`X_tr, X_val, X_te`, with the respective labels).

## Educational purpose

Learn the basic boosting workflow (low learning rate, many rounds as a ceiling, early stopping on the validation set) and understand that **boosting does not always win**: on noisy data a forest with the defaults can do just as well.

## Task

1. Model: `xgb.XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=3, eval_metric="logloss", early_stopping_rounds=30, n_jobs=-1, random_state=0)`, trained with `eval_set=[(X_val, y_val)]`. Save `rounds_used` (`best_iteration + 1`), `es_auc` and `es_ll` (AUC and log-loss on the test set).
2. The same model **without** early stopping (1000 rounds, without `early_stopping_rounds` or `eval_set`): `ll_1000` (log-loss on the test set).
3. Forest `RandomForestClassifier(300, random_state=0, n_jobs=-1)`: `rf_auc`. Save `xgb_wins` (True if `es_auc > rf_auc`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import train_test_split

X, y = make_classification(3000, n_features=12, n_informative=5, class_sep=0.8, flip_y=0.05, random_state=0)
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.4, random_state=0)
X_val, X_te, y_val, y_te = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0)

## Your work

In [ ]:
model = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=3, eval_metric="logloss",
                          early_stopping_rounds=30, n_jobs=-1, random_state=0)
model.fit(X_tr, y_tr, eval_set=[(..., ...)], verbose=False)
rounds_used = ...
p = model.predict_proba(X_te)[:, 1]
es_auc, es_ll = roc_auc_score(y_te, p), log_loss(y_te, p)

In [ ]:
no_es = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=3, eval_metric="logloss", n_jobs=-1, random_state=0).fit(X_tr, y_tr)
ll_1000 = ...

In [ ]:
rf = RandomForestClassifier(300, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
rf_auc = ...
xgb_wins = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`eval_set=[(X_val, y_val)]`; `model.best_iteration + 1`; `log_loss(y_te, no_es.predict_proba(X_te)[:, 1])`.'):
    check(50 < rounds_used < 500 and es_auc > 0.9, "early stopping halts between 50 and 500 rounds (out of 1000) and the AUC exceeds 0.9", "model.best_iteration + 1")
    check(es_ll < ll_1000 - 0.01, "with early stopping the test log-loss is better by over 0.01 than at 1000 rounds", "log_loss(y_te, no_es.predict_proba(X_te)[:, 1])")
    check(0.9 < rf_auc < 1.0 and xgb_wins in (True, False), "forest AUC between 0.9 and 1; `xgb_wins` is a boolean", "roc_auc_score(y_te, rf.predict_proba(X_te)[:, 1])")
done()

---
The commented solution is in `solutions/B_with_sklearn/R9_warmup_the_first_xgboost_and_early_stopping.ipynb`. Open it after you pass the check, or after you have really tried.